# CMP 170HX experiment ledger

**STATUS — PUBLICATION (measurements complete)**  
Last completed gate: measurements PASS — attempt 12 startup PASS (48/48 main shards + draft loaded, CUDA graph capture complete, /health 200), then cold/warm text smoke, TTFT, sustained decode, uncached prefill, and loaded telemetry receipts captured. Vision gate rejected: text-only serve path on the SM80 fork.  
Current command: publish receipts, README, and benchmark card; ClipProxy wiring stays open.  
Blocker: none for the text-only result; vision remains unsupported in the SM80 fork.  
Next gate: ClipProxy route wiring and live end-to-end verification.

## Attempt table

| Attempt | Phase | Single change | Expected signal | Stop condition | Result | Evidence |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | Import | Source-built SM80 fork | CUDA/custom ops/model registry pass | First decisive import error | PASS | `results/receipts/import-gate.json` |
| 2 | Load | Add F8_E8M0 safetensors mapping | 48 shards + KV allocation + ready | Ready or first decisive failure | Superseded — shard stream passed, engine readiness reached at attempt 12 | `results/receipts/load-gate.json` |
| 12 | Load | Bind-mount patched dspark.py draft loader | Ready + /health 200 after 48/48 shards + draft load | First decisive failure or readiness | PASS | `results/receipts/attempt-12-startup.json` |

Attempt 11 (draft-loader key error) and intermediate attempts are receipted in
`results/receipts/`; measurements for the attempt 12 server are summarized in
`results/receipts/measurements.json`.

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
PHASE_ORDER = ['identity', 'storage_preflight', 'load_gate', 'functional_gates', 'measurements', 'publication']
GPU_EXECUTION_ENABLED = os.environ.get('PIXELML_ENABLE_GPU_CELLS') == '1'

def read_json(relative: str) -> dict:
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

def require_phase(phase_id: str) -> dict:
    state = read_json('results/phase-status.json')
    statuses = {item['id']: item['status'] for item in state['phases']}
    target = PHASE_ORDER.index(phase_id)
    allowed = {'PASS', 'COMPLETE'}
    for prior in PHASE_ORDER[:target]:
        assert statuses[prior] in allowed, f'{prior} blocks {phase_id}: {statuses[prior]}'
    return state

def require_gpu_opt_in() -> None:
    assert GPU_EXECUTION_ENABLED, 'Set PIXELML_ENABLE_GPU_CELLS=1 only after ownership and preflight GO.'


## 1. Identity


In [ ]:
manifest = read_json('results/run-manifest.json')
assert manifest['model']['revision'] == '86f746b36186f0e567729a5c06a8c918caba82a9'
manifest


## 2. Storage + preflight


In [ ]:
preflight = read_json('results/receipts/preflight.json')
assert preflight['status'] == 'PASS'
assert preflight['go_no_go'] == 'GO_BOUNDED_COMPATIBILITY_GATE'
preflight


## 3. Load gate


In [ ]:
require_phase('load_gate')
import_gate = read_json('results/receipts/import-gate.json')
load_gate = read_json('results/receipts/load-gate.json')
assert import_gate['status'] == 'PASS'
load_gate


## 4. Functional gates
Later GPU cells remain blocked until the load gate is terminal PASS.


In [ ]:
try:
    require_phase('functional_gates')
    functional_state = 'READY'
except AssertionError as exc:
    functional_state = f'BLOCKED: {exc}'
functional_state


## 5. Measurements
Protocol: uncached prefill, TTFT, and decode at concurrency 1/2/4/6/8 with final-usage token counts.


In [ ]:
try:
    require_phase('measurements')
    measurement_state = 'READY'
except AssertionError as exc:
    measurement_state = f'BLOCKED: {exc}'
measurement_state


## 6. Publication
Measurements are complete; publication deliverables (receipts, README, HTML/PNG
benchmark card) land in this pass. ClipProxy route wiring and live end-to-end
verification remain the open follow-up gate.

In [ ]:
try:
    require_phase('publication')
    publication_state = 'READY'
except AssertionError as exc:
    publication_state = f'BLOCKED: {exc}'
publication_state
